In [ ]:
!pip install firecrawl-py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 kB 2.6 MB/s eta 0:00:00


In [ ]:
!pip install beautifulsoup4

# Setup

In [ ]:
import os
import base64
import pprint
import requests
import pandas as pd
import time
import base64
import numpy as np

from firecrawl import Firecrawl
from google.colab import userdata
from bs4 import BeautifulSoup

ModuleNotFoundError: No module named 'firecrawl'

In [ ]:
os.environ["FIRECRAWL_API_KEY"] = userdata.get("FIRECRAWL_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

firecrawl = Firecrawl(api_key=os.environ.get("FIRECRAWL_API_KEY"))

In [ ]:
os.mkdir("/content/train_raw_html")
os.mkdir("/content/train_image")
os.mkdir("/content/test_raw_html")
os.mkdir("/content/test_image")

# Util functions

In [ ]:
def get_full_url(short_url):
  try:
      response = requests.get(short_url, allow_redirects=True, timeout=10)

  except requests.exceptions.RequestException as e:
      print(f"An error occurred: {e}")
      return None

  return response.url

In [ ]:
def scrape_raw_html(url, output_dir):
  scrape_result = firecrawl.scrape(url, formats=['rawHtml'])

  with open(output_dir, "w") as f:
    f.write(scrape_result.raw_html)

In [ ]:
def scrape_image(question_id, input_dir, output_dir):
  with open(input_dir, "r") as f:
    html_content = f.read()

  soup = BeautifulSoup(html_content, "html.parser")
  status = False

  print(f"Processing question {question_id}")

  # Find the div with the image
  image_div = soup.find("div", class_="figures iviewerimage-parent")

  if image_div:
    target_image_element = image_div.find("div", class_=lambda c: c and "question__image iviewerimage" in c)

    if target_image_element:
      image_url = target_image_element.get("alt")
    else:
      image_url = None


    if image_url:
      print(f"Question {question_id} has image URL: {image_url}")

      response = requests.get(image_url)

      if response.status_code == 200:
        with open(output_dir, "wb") as img_file:
            img_file.write(response.content)
        print(f"Image downloaded")
        status = True

      else:
          print(f"Failed to download. Status code: {response.status_code}")
  else:
    print(f"Question {question_id} has no image")

  return status

In [ ]:
def image_to_base64(image_path):
    try:
        with open(image_path, "rb") as image_file:
            image_data = image_file.read()

            encoded_bytes = base64.b64encode(image_data)

            base64_string = encoded_bytes.decode('utf-8')

            print(f"Image converted to base64 successfully")
            return base64_string

    except FileNotFoundError:
        print(f"Error: The file '{image_path}' was not found.")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

In [ ]:
import os

def count_files_walk(directory_path):
    count = 0
    for root, dirs, files in os.walk(directory_path):
        count += len(files)
    return count

In [ ]:
# Login using e.g. `huggingface-cli login` to access this dataset
splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}

# Process train set
train_original_df = pd.read_parquet("hf://datasets/LangAGI-Lab/medbullets/" + splits["train"])

# Process test set
test_original_df = pd.read_parquet("hf://datasets/LangAGI-Lab/medbullets/" + splits["test"])
# test_original_df = test_original_df.drop(columns=["opa", "opb", "opc", "opd", "answer_idx", "ope"])

In [ ]:
# Get full link for train set
for i in range(len(train_original_df)):
  link = train_original_df.iloc[i]['link']

  if not (link.startswith('https://step2.medbullets.com')):
    link = get_full_url(link)

  train_original_df.loc[i, 'link'] = link

In [ ]:
# Get full link for test set
for i in range(len(test_original_df)):
  link = test_original_df.iloc[i]['link']

  if not (link.startswith('https://step2.medbullets.com')):
    link = get_full_url(link)

  test_original_df.loc[i, 'link'] = link

# Load dataset and EDA

In [ ]:
print(train_original_df.shape)
print(test_original_df.shape)

(492, 10)
(124, 10)


In [ ]:
train_original_df.to_csv("/content/train_original.csv")
test_original_df.to_csv("/content/test_original.csv")

In [ ]:
train_deduplicated_df = train_original_df.drop_duplicates(subset=['link'])
test_deduplicated_df = test_original_df.drop_duplicates(subset=['link'])

In [ ]:
print(train_deduplicated_df.shape)
print(test_deduplicated_df.shape)

(291, 4)
(116, 4)


In [ ]:
train_deduplicated_df.to_csv("/content/train_deduplicated.csv")
test_deduplicated_df.to_csv("/content/test_deduplicated.csv")

In [ ]:
train_image_df = pd.DataFrame(columns=["link", "question", "opa", "opb", "opc", "opd", "ope", "answer_idx", "answer", "explanation", "image_base64"])

In [ ]:
test_image_df = pd.DataFrame(columns=["link", "question", "opa", "opb", "opc", "opd", "ope", "answer_idx", "answer", "explanation", "image_base64"])

In [ ]:
print(train_image_df.shape)
print(test_image_df.shape)

(0, 5)
(0, 5)


# Train data

In [ ]:
# Process train data
track_count = 0

for i in range(len(train_deduplicated_df)):
  link = train_deduplicated_df.iloc[i]['link']

  question_id = link.split('=')[1]

  scrape_raw_html(
      url = link,
      output_dir = f"/content/train_raw_html/{question_id}.html"
  )

  status = scrape_image(
      question_id = question_id,
      input_dir = f"/content/train_raw_html/{question_id}.html",
      output_dir = f"/content/train_image/{question_id}.jpg"
  )

  if status:
    question = train_deduplicated_df.iloc[i]['question']
    answer = train_deduplicated_df.iloc[i]['answer']
    explanation = train_deduplicated_df.iloc[i]['explanation']
    opa = train_deduplicated_df.iloc[i]['opa']
    opb = train_deduplicated_df.iloc[i]['opb']
    opc = train_deduplicated_df.iloc[i]['opc']
    opd = train_deduplicated_df.iloc[i]['opd']
    ope = train_deduplicated_df.iloc[i]['ope']
    answer_idx = train_deduplicated_df.iloc[i]['answer_idx']
    image_base64 = image_to_base64(f"/content/train_image/{question_id}.jpg")

    if (image_base64 is not None):
      row = {'link': link, 'question': question, 'opa': opa, 'opb': opb, 'opc': opc, 'opd': opd, 'ope': ope, 'answer_idx': answer_idx, 'answer': answer, 'explanation': explanation, 'image_base64': image_base64}
      train_image_df.loc[len(train_image_df)] = row
      print(f"Question {question_id} added")

  track_count += 1
  print(f"Processed {track_count} questions")
  print()
  time.sleep(4)

Processing question 109022
Question 109022 has image URL: https://upload.medbullets.com/question/109022/images/subarachnoid%20hemorrhage.jpg
Image downloaded
Image converted to base64 successfully
Question 109022 added
Processed 1 questions

Processing question 109119
Question 109119 has no image
Processed 2 questions

Processing question 216501
Question 216501 has no image
Processed 3 questions

Processing question 109937
Question 109937 has no image
Processed 4 questions

Processing question 216438
Question 216438 has no image
Processed 5 questions

Processing question 109754
Question 109754 has no image
Processed 6 questions

Processing question 109208
Question 109208 has image URL: https://upload.medbullets.com/question/109208/images/normal%20ultrasound.jpg
Image downloaded
Image converted to base64 successfully
Question 109208 added
Processed 7 questions

Processing question 109241
Question 109241 has no image
Processed 8 questions

Processing question 109961
Question 109961 has n

In [ ]:
train_image_df['image_base64'] = train_image_df['image_base64'].replace(['', ' '], np.nan)
train_image_df.dropna(subset=['image_base64'], inplace=True)

In [ ]:
print(train_image_df.shape)

(113, 5)


In [ ]:
train_image_df.to_csv('/content/train_image.csv')

In [ ]:
# Example usage:
path_to_scan = '/content/train_raw_html'
file_count = count_files_walk(path_to_scan)
print(f"Number of files (recursive) in {path_to_scan}: {file_count}")

Number of files (recursive) in /content/train_raw_html: 291


In [ ]:
# Example usage:
path_to_scan = '/content/train_image'
file_count = count_files_walk(path_to_scan)
print(f"Number of files (recursive) in {path_to_scan}: {file_count}")

Number of files (recursive) in /content/train_image: 113


In [ ]:
import os
from google.colab import files

# Specify the folder to zip
folder_to_zip = '/content/train_image'
output_zip_name = 'train_image.zip'

# Create the zip file
!zip -r {output_zip_name} {folder_to_zip}

# Download the zip file
files.download(output_zip_name)

  adding: content/train_image/ (stored 0%)
  adding: content/train_image/215171.jpg (deflated 6%)
  adding: content/train_image/216508.jpg (deflated 0%)
  adding: content/train_image/109240.jpg (deflated 8%)
  adding: content/train_image/216259.jpg (deflated 8%)
  adding: content/train_image/104911.jpg (deflated 0%)
  adding: content/train_image/215054.jpg (deflated 70%)
  adding: content/train_image/214945.jpg (deflated 2%)
  adding: content/train_image/109433.jpg (deflated 3%)
  adding: content/train_image/108933.jpg (deflated 6%)
  adding: content/train_image/216637.jpg (deflated 1%)
  adding: content/train_image/109249.jpg (deflated 8%)
  adding: content/train_image/109254.jpg (deflated 2%)
  adding: content/train_image/215049.jpg (deflated 1%)
  adding: content/train_image/109282.jpg (deflated 0%)
  adding: content/train_image/216626.jpg (deflated 0%)
  adding: content/train_image/109958.jpg (deflated 0%)
  adding: content/train_image/210867.jpg (deflated 1%)
  adding: content/tra

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Test data

In [ ]:
# Process test data
track_count = 0

for i in range(len(test_deduplicated_df)):
  link = test_deduplicated_df.iloc[i]['link']

  question_id = link.split('=')[1]

  scrape_raw_html(
      url = link,
      output_dir = f"/content/test_raw_html/{question_id}.html"
  )

  status = scrape_image(
      question_id = question_id,
      input_dir = f"/content/test_raw_html/{question_id}.html",
      output_dir = f"/content/test_image/{question_id}.jpg"
  )

  if status:
    question = test_deduplicated_df.iloc[i]['question']
    answer = test_deduplicated_df.iloc[i]['answer']
    explanation = test_deduplicated_df.iloc[i]['explanation']
    opa = test_deduplicated_df.iloc[i]['opa']
    opb = test_deduplicated_df.iloc[i]['opb']
    opc = test_deduplicated_df.iloc[i]['opc']
    opd = test_deduplicated_df.iloc[i]['opd']
    ope = test_deduplicated_df.iloc[i]['ope']
    answer_idx = test_deduplicated_df.iloc[i]['answer_idx']
    image_base64 = image_to_base64(f"/content/test_image/{question_id}.jpg")

    if (image_base64 is not None):
      row = {'link': link, 'question': question, 'opa': opa, 'opb': opb, 'opc': opc, 'opd': opd, 'ope': ope, 'answer_idx': answer_idx, 'answer': answer, 'explanation': explanation, 'image_base64': image_base64}
      test_image_df.loc[len(test_image_df)] = row
      print(f"Question {question_id} added")

  track_count += 1
  print(f"Processed {track_count} questions")
  print()
  time.sleep(6)

Processing question 108918
Question 108918 has no image
Processed 1 questions

Processing question 210697
Question 210697 has image URL: https://upload.medbullets.com/question/210697/images/meninrash..jpg
Image downloaded
Image converted to base64 successfully
Question 210697 added
Processed 2 questions

Processing question 216625
Question 216625 has image URL: https://upload.medbullets.com/question/216625/images/shortqt.jpg
Image downloaded
Image converted to base64 successfully
Question 216625 added
Processed 3 questions

Processing question 216239
Question 216239 has no image
Processed 4 questions

Processing question 108912
Question 108912 has image URL: https://upload.medbullets.com/question/108912/images/mobitz%202.jpg
Image downloaded
Image converted to base64 successfully
Question 108912 added
Processed 5 questions

Processing question 217175
Question 217175 has no image
Processed 6 questions

Processing question 109008
Question 109008 has no image
Processed 7 questions

Proces

In [ ]:
test_image_df['image_base64'] = test_image_df['image_base64'].replace(['', ' '], np.nan)
test_image_df.dropna(subset=['image_base64'], inplace=True)

In [ ]:
print(test_image_df.shape)

(47, 5)


In [ ]:
test_image_df.to_csv('/content/test_image.csv')

In [ ]:
# Example usage:
path_to_scan = '/content/test_raw_html'
file_count = count_files_walk(path_to_scan)
print(f"Number of files (recursive) in {path_to_scan}: {file_count}")

Number of files (recursive) in /content/test_raw_html: 116


In [ ]:
# Example usage:
path_to_scan = '/content/test_image'
file_count = count_files_walk(path_to_scan)
print(f"Number of files (recursive) in {path_to_scan}: {file_count}")

Number of files (recursive) in /content/test_image: 47


In [ ]:
import os
from google.colab import files

# Specify the folder to zip
folder_to_zip = '/content/test_image'
output_zip_name = 'test_image.zip'

# Create the zip file
!zip -r {output_zip_name} {folder_to_zip}

# Download the zip file
files.download(output_zip_name)

  adding: content/test_image/ (stored 0%)
  adding: content/test_image/215171.jpg (deflated 6%)
  adding: content/test_image/216259.jpg (deflated 8%)
  adding: content/test_image/109433.jpg (deflated 3%)
  adding: content/test_image/108933.jpg (deflated 6%)
  adding: content/test_image/216637.jpg (deflated 1%)
  adding: content/test_image/109249.jpg (deflated 8%)
  adding: content/test_image/109282.jpg (deflated 0%)
  adding: content/test_image/210867.jpg (deflated 1%)
  adding: content/test_image/210369.jpg (deflated 14%)
  adding: content/test_image/108988.jpg (deflated 2%)
  adding: content/test_image/108992.jpg (deflated 1%)
  adding: content/test_image/216587.jpg (deflated 37%)
  adding: content/test_image/109041.jpg (deflated 4%)
  adding: content/test_image/109919.jpg (deflated 1%)
  adding: content/test_image/216634.jpg (deflated 1%)
  adding: content/test_image/109964.jpg (deflated 0%)
  adding: content/test_image/214944.jpg (deflated 1%)
  adding: content/test_image/215176.jp

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>